# LEO Satellite Routing — Results Walkthrough

This notebook walks through the key results from the paper:
> *Mimicking Delay-Aware Routing in LEO Networks: A Random Forest Approach with Local-Only Features*

Three routing policies are compared on the same source-destination pairs:
- **DistanceOnly** — Dijkstra with geometric distance. Fast, simple, blind to congestion.
- **DelayAware** — Dijkstra with queue-aware cost. The oracle. Needs global state.
- **ML-Agent** — Random Forest trained on oracle labels. Runs locally, no global state needed.

All numbers are from paired evaluation on a 90-node OneWeb-like constellation under European-peak diurnal Poisson traffic.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f9f9f9',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'font.size': 12,
    'axes.spines.top': False,
    'axes.spines.right': False
})

COLORS = {
    'DelayAware':   '#2ecc71',
    'ML-Agent':     '#3498db',
    'DistanceOnly': '#e74c3c'
}

print('Libraries loaded.')

## 1 — End-to-end delay

The most important metric. Under European-peak congestion, distance-only routing saturates the popular corridors and delay explodes. The ML agent — using only local queue state and one-hop neighbors — tracks the oracle almost exactly.

In [ ]:
# Results from Table V — online routing on ML evaluation subset
policies = ['DelayAware', 'ML-Agent', 'DistanceOnly']

median_delay = {'DelayAware': 1.232, 'ML-Agent': 1.256, 'DistanceOnly': 8.093}
mean_delay   = {'DelayAware': 1.41,  'ML-Agent': 1.63,  'DistanceOnly': 9.21}

fig, ax = plt.subplots(figsize=(8, 4))

x = np.arange(len(policies))
bars = ax.bar(x, [median_delay[p] for p in policies],
              color=[COLORS[p] for p in policies],
              width=0.5, zorder=3)

for bar, policy in zip(bars, policies):
    val = median_delay[policy]
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.15,
            f'{val:.3f} s', ha='center', va='bottom', fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(policies)
ax.set_ylabel('Median end-to-end delay (s)')
ax.set_title('End-to-end delay — lower is better')
ax.set_ylim(0, 10)

ax.annotate('', xy=(1, 1.256), xytext=(0, 1.232),
            arrowprops=dict(arrowstyle='<->', color='gray', lw=1.2))
ax.text(0.5, 1.8, 'gap: 0.024 s', ha='center', color='gray', fontsize=10)

plt.tight_layout()
plt.savefig('delay_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'ML-Agent recovers {((8.093-1.256)/(8.093-1.232))*100:.1f}% of the oracle improvement over baseline.')

## 2 — Path drop rate

DistanceOnly routes through the geometrically shortest paths — which are also the most congested. Buffers overflow. About 7% of packets never arrive.

Both DelayAware and the ML-Agent keep drop rate at exactly 0% on the evaluation subset. This is the most operationally important result.

In [ ]:
drop_rates = {'DelayAware': 0.0, 'ML-Agent': 0.0, 'DistanceOnly': 6.95}

fig, ax = plt.subplots(figsize=(8, 4))

bars = ax.bar(np.arange(len(policies)),
              [drop_rates[p] for p in policies],
              color=[COLORS[p] for p in policies],
              width=0.5, zorder=3)

for bar, policy in zip(bars, policies):
    val = drop_rates[policy]
    label = f'{val:.2f}%' if val > 0 else '0%'
    ax.text(bar.get_x() + bar.get_width()/2,
            val + 0.1, label, ha='center', va='bottom', fontweight='bold')

ax.set_xticks(np.arange(len(policies)))
ax.set_xticklabels(policies)
ax.set_ylabel('Mean path drop rate (%)')
ax.set_title('Path drop rate — lower is better')
ax.set_ylim(0, 10)

plt.tight_layout()
plt.savefig('drop_rate_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 3 — Feature importance

Which features actually drive the routing decisions?

The ablation study in the paper gives a clear hierarchy. Removing the neighbor queue proxy pushes the agent all the way back toward distance-only behavior. The congestion flag catches transient bursts. Geometric progress acts as a tiebreaker. Propagation alone is not enough.

In [ ]:
# Relative importance from ablation results (Section VI-E)
features = [
    'Neighbor queue proxy\n(Q(t)/µ)',
    'Congestion flag\n(busy threshold)',
    'Geometric progress\n(Δ distance to dest)',
    'Link propagation\n(distance/c)',
    "Source queue ratio",
    'Queue trend\n(Δ over time)'
]
importance = [0.41, 0.22, 0.14, 0.10, 0.08, 0.05]

fig, ax = plt.subplots(figsize=(8, 5))

y = np.arange(len(features))
bars = ax.barh(y, importance, color='#3498db', alpha=0.85, zorder=3)

for bar, val in zip(bars, importance):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.0%}', va='center', fontsize=10)

ax.set_yticks(y)
ax.set_yticklabels(features, fontsize=10)
ax.set_xlabel('Relative importance (from ablation study)')
ax.set_title('Feature importance — what drives routing decisions')
ax.set_xlim(0, 0.55)
ax.invert_yaxis()

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Ablation hierarchy: queue proxy >> congestion flag > geometric progress > propagation alone')

## 4 — Offline classification quality

How well does the student reproduce the teacher's exact next-hop choices on held-out decision groups?

The Y-randomization control is the honest part: when labels are shuffled, accuracy drops from 0.81 to 0.16. This confirms the model learned real routing structure — not a statistical artifact.

In [ ]:
# Table III — mean ± std across 10 random seeds
metrics = {
    'Group Top-1\n(main metric)': (0.809, 0.041),
    'Balanced\naccuracy':         (0.831, 0.037),
    'Precision\n(label=1)':       (0.804, 0.068),
    'Recall\n(label=1)':          (0.678, 0.073),
    'Out-of-bag\nscore':          (0.944, 0.003),
    'Y-randomization\n(control)': (0.164, 0.072),
}

labels = list(metrics.keys())
means  = [v[0] for v in metrics.values()]
stds   = [v[1] for v in metrics.values()]
colors = ['#e74c3c' if 'rand' in l.lower() else '#3498db' for l in labels]

fig, ax = plt.subplots(figsize=(10, 4))

x = np.arange(len(labels))
bars = ax.bar(x, means, yerr=stds, color=colors, width=0.55,
              capsize=5, zorder=3, error_kw={'linewidth': 1.5})

for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, mean + 0.02,
            f'{mean:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.axhline(0.5, color='gray', linestyle='--', linewidth=1, alpha=0.6, label='chance level')
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('Score (mean ± std, 10 seeds)')
ax.set_title('Classification quality — bars show mean ± std across 10 random seeds')
ax.set_ylim(0, 1.1)

real = mpatches.Patch(color='#3498db', label='Real model')
rand = mpatches.Patch(color='#e74c3c', label='Y-randomization (shuffled labels)')
ax.legend(handles=[real, rand], loc='upper right', fontsize=9)

plt.tight_layout()
plt.savefig('classification_quality.png', dpi=150, bbox_inches='tight')
plt.show()
print('Y-randomization: Wilcoxon p=0.002, Cliff delta=1.0 — model learns real structure.')

## 5 — The honest result: temporal generalization

What happens when you train on early traffic snapshots and test on peak-hour snapshots the model never saw?

Offline classification accuracy drops from 0.81 to 0.41. This is a real limitation — static-snapshot training doesn't automatically transfer to shifted traffic distributions. Periodic retraining would be needed in deployment.

That said, online end-to-end routing performance stays close to the oracle even in this protocol — because routing only needs the top-ranked candidate per hop, and near-tie errors don't change the path much.

In [ ]:
# Table IV — in-sample vs held-out time protocol
protocols   = ['In-sample\n(group-aware 80/20)', 'Held-out time\n(train t≤90, test t≥120)']
top1_mean   = [0.809, 0.406]
top1_std    = [0.041, 0.008]
bal_mean    = [0.831, 0.616]
bal_std     = [0.037, 0.009]

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

for ax, means, stds, title in [
    (axes[0], top1_mean, top1_std, 'Group Top-1 accuracy'),
    (axes[1], bal_mean,  bal_std,  'Balanced accuracy')
]:
    colors = ['#3498db', '#e67e22']
    bars = ax.bar(np.arange(2), means, yerr=stds, color=colors,
                  width=0.45, capsize=6, zorder=3,
                  error_kw={'linewidth': 1.5})
    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, mean + 0.015,
                f'{mean:.3f}', ha='center', va='bottom', fontweight='bold')
    ax.set_xticks([0, 1])
    ax.set_xticklabels(protocols, fontsize=9)
    ax.set_title(title)
    ax.set_ylim(0, 1.05)
    ax.axhline(0.5, color='gray', linestyle='--', linewidth=1, alpha=0.5)

axes[0].set_ylabel('Score (mean ± std, 10 seeds)')
fig.suptitle('Generalization under temporal distribution shift', fontweight='bold')

plt.tight_layout()
plt.savefig('temporal_generalization.png', dpi=150, bbox_inches='tight')
plt.show()
print('78% of test (s,d) pairs completely unseen during training.')
print('Variance collapses in held-out protocol (std 0.008 vs 0.041) — drop is stable, not a bad split.')

## Summary

| | ML-Agent | Oracle | Baseline |
|---|---|---|---|
| Median delay | 1.26 s | 1.23 s | 8.09 s |
| Drop rate | 0% | 0% | ~6.95% |
| Group Top-1 (in-sample) | 0.809 ± 0.041 | — | — |
| Group Top-1 (held-out time) | 0.406 ± 0.008 | — | — |

A Random Forest using only near-local features recovers ~99.7% of the oracle's delay improvement over the geometric baseline — without any inter-satellite queue exchange at runtime. The held-out time result is a known limitation and motivates periodic retraining in any real deployment.